<a href="https://colab.research.google.com/github/dJasawat/ByNethaji_DeepLearing_Notebooks/blob/main/filesystem_mcp_colab_corrected.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📁 Filesystem MCP — Corrected Google Colab Notebook

This notebook fixes the Colab error:

```text
UnsupportedOperation: fileno
```

## Why the original notebook failed

The MCP Python client's `stdio_client(...)` starts the MCP server as a subprocess and, by default, forwards the subprocess's `stderr` to `sys.stderr`.

In Google Colab/Jupyter, `sys.stderr` is usually an IPython `OutStream`, not a normal operating-system file. It therefore may not provide a usable `fileno()`. Subprocess creation then fails before the MCP initialize handshake completes.

## Correct fix used here

Instead of replacing `sys.stderr` globally, this notebook explicitly gives `stdio_client` a real log file:

```python
with open("mcp_server_stderr.log", "a") as errlog:
    async with stdio_client(server_params, errlog=errlog):
        ...
```

The notebook first tests MCP directly **without an LLM**, and then runs both:

1. Manual tool-schema mode
2. Dynamic MCP tool-discovery mode

Filesystem access is restricted to `mcp_workspace/` for safer experimentation.


In [ ]:
# Install the current MCP v1 line used by this notebook.
# The <2 bound matters because MCP 2.x is a separate SDK generation.
%pip install -q --upgrade "mcp>=1.27,<2" "openai>=1.75.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.8 MB/s eta 0:00:00


## 1. Create a safe filesystem workspace

In [ ]:
import os
import sys
import io
from pathlib import Path

ROOT_DIR = Path.cwd().resolve()
WORKSPACE = (ROOT_DIR / "mcp_workspace").resolve()
SERVER_PATH = ROOT_DIR / "mcp_filesystem_server.py"
MCP_LOG_PATH = ROOT_DIR / "mcp_server_stderr.log"

WORKSPACE.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "hello.txt").write_text(
    "Hello from the Colab filesystem MCP experiment!\n",
    encoding="utf-8",
)
(WORKSPACE / "notes").mkdir(exist_ok=True)
(WORKSPACE / "notes" / "mcp.txt").write_text(
    "MCP separates tool providers (servers) from tool users (clients).\n",
    encoding="utf-8",
)

print("Python executable:", sys.executable)
print("Notebook root:    ", ROOT_DIR)
print("Safe workspace:   ", WORKSPACE)
print("Server log:       ", MCP_LOG_PATH)


Python executable: /usr/bin/python3
Notebook root:     /content
Safe workspace:    /content/mcp_workspace
Server log:        /content/mcp_server_stderr.log


## 2. Confirm the Colab stream problem

In [ ]:
print("sys.stderr type:", type(sys.stderr).__name__)

try:
    print("sys.stderr file descriptor:", sys.stderr.fileno())
except (io.UnsupportedOperation, AttributeError) as exc:
    print("Expected in many Colab/Jupyter runtimes:")
    print(f"  {type(exc).__name__}: {exc}")

# A normal disk file has the real OS descriptor required by subprocess creation.
with MCP_LOG_PATH.open("a", encoding="utf-8") as real_log:
    print("Real log file descriptor:", real_log.fileno())

print("✅ We will pass this real file explicitly to stdio_client(errlog=...).")


sys.stderr type: OutStream
Expected in many Colab/Jupyter runtimes:
  UnsupportedOperation: fileno
Real log file descriptor: 52
✅ We will pass this real file explicitly to stdio_client(errlog=...).


## 3. Generate the FastMCP filesystem server

In [ ]:
%%writefile mcp_filesystem_server.py
from __future__ import annotations

import os
from datetime import datetime, timezone
from pathlib import Path

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("ColabFilesystem")
WORKSPACE = Path(os.environ["MCP_WORKSPACE"]).resolve()


def safe_path(user_path: str = ".") -> Path:
    """Resolve a user path and prevent access outside WORKSPACE."""
    candidate = Path(user_path).expanduser()
    if not candidate.is_absolute():
        candidate = WORKSPACE / candidate

    resolved = candidate.resolve()

    if resolved != WORKSPACE and WORKSPACE not in resolved.parents:
        raise ValueError(
            f"Access denied: '{user_path}' is outside the MCP workspace."
        )
    return resolved


def display_path(path: Path) -> str:
    """Return a path relative to the workspace when possible."""
    try:
        relative = path.relative_to(WORKSPACE)
        return "." if str(relative) == "." else str(relative)
    except ValueError:
        return str(path)


@mcp.tool()
def list_directory(path: str = ".") -> str:
    """List files and folders inside a workspace directory."""
    try:
        directory = safe_path(path)
        if not directory.exists():
            return f"Error: directory does not exist: {display_path(directory)}"
        if not directory.is_dir():
            return f"Error: path is not a directory: {display_path(directory)}"

        rows = []
        for item in sorted(directory.iterdir(), key=lambda p: (not p.is_dir(), p.name.lower())):
            tag = "[DIR]" if item.is_dir() else "[FILE]"
            rows.append(f"{tag} {item.name}")

        return "\n".join(rows) if rows else "Directory is empty."
    except Exception as exc:
        return f"Error listing directory '{path}': {type(exc).__name__}: {exc}"


@mcp.tool()
def read_file(filepath: str) -> str:
    """Read a UTF-8 text file from the workspace."""
    try:
        path = safe_path(filepath)
        if not path.exists():
            return f"Error: file does not exist: {display_path(path)}"
        if not path.is_file():
            return f"Error: path is not a file: {display_path(path)}"
        return path.read_text(encoding="utf-8")
    except Exception as exc:
        return f"Error reading '{filepath}': {type(exc).__name__}: {exc}"


@mcp.tool()
def write_file(filepath: str, content: str) -> str:
    """Create or overwrite a UTF-8 text file inside the workspace."""
    try:
        path = safe_path(filepath)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content, encoding="utf-8")
        return (
            f"Successfully wrote {len(content)} characters to "
            f"'{display_path(path)}'."
        )
    except Exception as exc:
        return f"Error writing '{filepath}': {type(exc).__name__}: {exc}"


@mcp.tool()
def get_file_info(filepath: str) -> str:
    """Return type, size and modification time for a workspace path."""
    try:
        path = safe_path(filepath)
        stat = path.stat()
        kind = "Directory" if path.is_dir() else "File"
        modified = datetime.fromtimestamp(
            stat.st_mtime, tz=timezone.utc
        ).isoformat()

        return (
            f"Path: {display_path(path)}\n"
            f"Type: {kind}\n"
            f"Size: {stat.st_size} bytes\n"
            f"Modified (UTC): {modified}"
        )
    except Exception as exc:
        return f"Error inspecting '{filepath}': {type(exc).__name__}: {exc}"


if __name__ == "__main__":
    # Explicit transport keeps the experiment clear.
    mcp.run(transport="stdio")


Writing mcp_filesystem_server.py


In [ ]:
# Syntax-check the generated server before attempting MCP startup.
import py_compile

py_compile.compile(str(SERVER_PATH), doraise=True)
print("✅ Server script syntax is valid:", SERVER_PATH)


✅ Server script syntax is valid: /content/mcp_filesystem_server.py


## 4. Create a Colab-safe MCP client session

In [ ]:
import json
from contextlib import asynccontextmanager

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Do not send the notebook's full environment to the server.
# In particular, an API key is not required by this filesystem subprocess.
server_params = StdioServerParameters(
    command=sys.executable,
    args=["-u", str(SERVER_PATH)],
    env={
        "MCP_WORKSPACE": str(WORKSPACE),
        "PYTHONUNBUFFERED": "1",
    },
    cwd=str(ROOT_DIR),
)


@asynccontextmanager
async def open_filesystem_session():
    """
    Open one MCP stdio session.

    Critical Colab fix:
    pass a real disk file as errlog instead of allowing stdio_client
    to use IPython's virtual sys.stderr stream.
    """
    MCP_LOG_PATH.touch(exist_ok=True)

    with MCP_LOG_PATH.open(
        "a",
        encoding="utf-8",
        buffering=1,
    ) as errlog:
        async with stdio_client(
            server_params,
            errlog=errlog,  # <-- fixes UnsupportedOperation: fileno
        ) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                yield session


def result_to_text(result) -> str:
    """Convert MCP text content blocks into one printable string."""
    text_blocks = [
        block.text
        for block in result.content
        if getattr(block, "type", None) == "text"
        and hasattr(block, "text")
    ]
    return "\n".join(text_blocks)


def show_server_log_tail(lines: int = 30) -> None:
    if not MCP_LOG_PATH.exists():
        print("No MCP server log exists yet.")
        return

    content = MCP_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    print(f"--- Last {min(lines, len(content))} MCP server log lines ---")
    print("\n".join(content[-lines:]) if content else "(log is empty)")


## 5. Core MCP smoke test — no LLM or API key

Run this first. It isolates the MCP connection from Groq/OpenAI tool-calling logic.


In [ ]:
async with open_filesystem_session() as session:
    tools_response = await session.list_tools()
    tool_names = [tool.name for tool in tools_response.tools]
    print("✅ MCP connected.")
    print("Discovered tools:", tool_names)

    result = await session.call_tool(
        "list_directory",
        arguments={"path": "."},
    )
    print("\nWorkspace contents:")
    print(result_to_text(result))


✅ MCP connected.
Discovered tools: ['list_directory', 'read_file', 'write_file', 'get_file_info']

Workspace contents:
[DIR] notes
[FILE] hello.txt


In [ ]:
# Direct write/read/info test through MCP
async with open_filesystem_session() as session:
    write_result = await session.call_tool(
        "write_file",
        arguments={
            "filepath": "direct_test.txt",
            "content": "This file was created through a Colab MCP stdio session.",
        },
    )
    print(result_to_text(write_result))

    read_result = await session.call_tool(
        "read_file",
        arguments={"filepath": "direct_test.txt"},
    )
    print("\nRead result:")
    print(result_to_text(read_result))

    info_result = await session.call_tool(
        "get_file_info",
        arguments={"filepath": "direct_test.txt"},
    )
    print("\nFile information:")
    print(result_to_text(info_result))


Successfully wrote 56 characters to 'direct_test.txt'.

Read result:
This file was created through a Colab MCP stdio session.

File information:
Path: direct_test.txt
Type: File
Size: 56 bytes
Modified (UTC): 2026-08-02T16:12:53.084382+00:00


## 6. Optional: connect Groq for the agent experiments

Recommended Colab setup:

1. Open the **Secrets** panel (key icon).
2. Add a secret named `GROQ_API_KEY`.
3. Enable notebook access for that secret.

The fallback uses `getpass`, so the key is not displayed or saved in the notebook.


In [ ]:
from getpass import getpass
from openai import AsyncOpenAI


def load_groq_api_key() -> str:
    key = os.environ.get("GROQ_API_KEY", "").strip()

    if not key:
        try:
            from google.colab import userdata
            key = (userdata.get("GROQ_API_KEY") or "").strip()
        except Exception:
            pass

    if not key:
        key = getpass("Enter GROQ_API_KEY (hidden): ").strip()

    if not key:
        raise ValueError("A Groq API key is required for the agent experiments.")

    return key


GROQ_API_KEY = load_groq_api_key()

llm_client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY,
)

MODEL_ID = "llama-3.3-70b-versatile"
print("✅ Groq client configured for:", MODEL_ID)


✅ Groq client configured for: llama-3.3-70b-versatile


## 7. Manual and dynamic tool-schema agent

In [ ]:
TOOLS_MANUAL = [
    {
        "type": "function",
        "function": {
            "name": "list_directory",
            "description": "List files and folders inside the MCP workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Workspace-relative directory path, such as '.'.",
                    }
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a UTF-8 text file inside the MCP workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {"type": "string"}
                },
                "required": ["filepath"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Create or overwrite a text file inside the MCP workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {"type": "string"},
                    "content": {"type": "string"},
                },
                "required": ["filepath", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_file_info",
            "description": "Get metadata for a file or folder inside the MCP workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {"type": "string"}
                },
                "required": ["filepath"],
            },
        },
    },
]


def dynamic_openai_tools(mcp_tools_response) -> list[dict]:
    """Convert MCP Tool definitions into OpenAI-compatible function tools."""
    return [
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description or "",
                "parameters": tool.inputSchema,
            },
        }
        for tool in mcp_tools_response.tools
    ]


async def run_filesystem_agent(
    question: str,
    mode: str = "dynamic",
    max_steps: int = 8,
) -> str:
    """
    Run a Groq tool-calling loop backed by the local MCP filesystem server.

    mode='manual'  -> use hardcoded OpenAI function schemas
    mode='dynamic' -> discover schemas from session.list_tools()
    """
    if mode not in {"manual", "dynamic"}:
        raise ValueError("mode must be 'manual' or 'dynamic'")

    try:
        async with open_filesystem_session() as session:
            print(f"🚀 MCP filesystem server connected. Mode: {mode}")

            if mode == "manual":
                tools = TOOLS_MANUAL
                print(f"📦 Loaded {len(tools)} manual schemas.")
            else:
                discovered = await session.list_tools()
                tools = dynamic_openai_tools(discovered)
                names = [tool["function"]["name"] for tool in tools]
                print(f"📦 Dynamically discovered {len(tools)} tools: {names}")

            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are a filesystem assistant. For every filesystem fact "
                        "or action, you must use the provided tools rather than guess. "
                        "All paths are relative to a safe workspace."
                    ),
                },
                {"role": "user", "content": question},
            ]

            for step in range(1, max_steps + 1):
                response = await llm_client.chat.completions.create(
                    model=MODEL_ID,
                    messages=messages,
                    tools=tools,
                    tool_choice="auto",
                    temperature=0.0,
                )

                assistant_message = response.choices[0].message

                if not assistant_message.tool_calls:
                    final_text = assistant_message.content or "(No text returned.)"
                    print("\n✨ FINAL AGENT RESPONSE")
                    print(final_text)
                    return final_text

                # Preserve the assistant tool-call message in the conversation.
                messages.append(
                    assistant_message.model_dump(exclude_none=True)
                )

                for tool_call in assistant_message.tool_calls:
                    tool_name = tool_call.function.name

                    try:
                        arguments = json.loads(
                            tool_call.function.arguments or "{}"
                        )
                    except json.JSONDecodeError as exc:
                        arguments = {}
                        tool_output = (
                            f"Invalid JSON arguments from model: {exc}"
                        )
                    else:
                        print(
                            f"[Step {step}] 🔧 {tool_name}({arguments})"
                        )
                        mcp_result = await session.call_tool(
                            tool_name,
                            arguments=arguments,
                        )
                        tool_output = result_to_text(mcp_result)
                        print(
                            f"[Step {step}] ✅ "
                            f"{tool_output[:300].strip()}"
                        )

                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "name": tool_name,
                            "content": tool_output,
                        }
                    )

            raise RuntimeError(
                f"Agent exceeded the maximum of {max_steps} tool steps."
            )

    except Exception:
        print("\n❌ Agent/MCP run failed. Server-log tail:")
        show_server_log_tail()
        raise
    finally:
        print("\n🛑 MCP session closed.")


## 8. Experiment 1 — manual schemas

In [ ]:
await run_filesystem_agent(
    "List all files and folders in the current MCP working directory.",
    mode="manual",
)


🚀 MCP filesystem server connected. Mode: manual
📦 Loaded 4 manual schemas.
[Step 1] 🔧 list_directory({'path': '.'})
[Step 1] ✅ [DIR] notes
[FILE] direct_test.txt
[FILE] hello.txt
[Step 2] 🔧 list_directory({'path': '.'})
[Step 2] ✅ [DIR] notes
[FILE] direct_test.txt
[FILE] hello.txt

✨ FINAL AGENT RESPONSE
The current directory contains two files: 'direct_test.txt' and 'hello.txt', and one folder: 'notes'.

🛑 MCP session closed.


"The current directory contains two files: 'direct_test.txt' and 'hello.txt', and one folder: 'notes'."

## 9. Experiment 2 — dynamic MCP discovery

In [ ]:
await run_filesystem_agent(
    "Create a file named 'agent_output.txt' containing a short explanation "
    "of the Model Context Protocol. Then read it and report its file size.",
    mode="dynamic",
)


🚀 MCP filesystem server connected. Mode: dynamic
📦 Dynamically discovered 4 tools: ['list_directory', 'read_file', 'write_file', 'get_file_info']
[Step 1] 🔧 write_file({'content': 'The Model Context Protocol is a set of rules governing the interaction between models and their environment.', 'filepath': 'agent_output.txt'})
[Step 1] ✅ Successfully wrote 108 characters to 'agent_output.txt'.
[Step 1] 🔧 read_file({'filepath': 'agent_output.txt'})
[Step 1] ✅ The Model Context Protocol is a set of rules governing the interaction between models and their environment.
[Step 1] 🔧 get_file_info({'filepath': 'agent_output.txt'})
[Step 1] ✅ Path: agent_output.txt
Type: File
Size: 108 bytes
Modified (UTC): 2026-08-02T16:13:04.598434+00:00

✨ FINAL AGENT RESPONSE
The file 'agent_output.txt' has been created with the content, read, and its size reported as 108 bytes.

🛑 MCP session closed.


"The file 'agent_output.txt' has been created with the content, read, and its size reported as 108 bytes."

## 10. Troubleshooting

In [ ]:
# Inspect server-side stderr only when a run fails or behaves unexpectedly.
show_server_log_tail(lines=50)


--- Last 26 MCP server log lines ---
[08/02/26 16:12:42] INFO     Processing request of type            server.py:733
                             ListToolsRequest                                   
                    INFO     Processing request of type            server.py:733
                             CallToolRequest                                    
[08/02/26 16:12:53] INFO     Processing request of type            server.py:733
                             CallToolRequest                                    
                    INFO     Processing request of type            server.py:733
                             ListToolsRequest                                   
                    INFO     Processing request of type            server.py:733
                             CallToolRequest                                    
                    INFO     Processing request of type            server.py:733
                             CallToolRequest                            

### What changed from the original notebook

- `stdio_client(server_params, errlog=real_log_file)` is now explicit.
- No global replacement of `sys.stderr` is required.
- A direct MCP test runs before any LLM call.
- MCP subprocess access is restricted to `mcp_workspace/`.
- The Groq API key is no longer hardcoded.
- The subprocess receives only the environment variables it needs.
- The server uses `python -u` and `PYTHONUNBUFFERED=1`.
- The client uses clean nested async context managers instead of manually closing an `AsyncExitStack`.
